In [ ]:
# Cell 1 of 2: setup. Clones the repo at main, installs the pinned packages, reads HF_TOKEN
# from Colab Secrets, checks the environment, and creates the Hub repos if they are missing.
# After a disconnect, reconnect and press Runtime > Run all. The run resumes where it stopped.
import os
import subprocess
import sys

NOTEBOOK = "02_train_explain"
REPO_URL = "https://github.com/Rohanjain2312/trade-settlement-fail-predictor.git"
CODE_DIR = "/content/trade-settlement-fail-predictor"

os.chdir("/content")
subprocess.run(["rm", "-rf", CODE_DIR], check=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, CODE_DIR], check=True)
os.chdir(CODE_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)

from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as err:
    raise SystemExit(
        "HF_TOKEN not found. Open the key icon on the left (Secrets), add HF_TOKEN with your "
        "Hugging Face write token, switch on notebook access, then press Run all again."
    ) from err
os.environ["TSFP_WORKDIR"] = "/content/tsfp_work"
subprocess.run([sys.executable, "-m", "src.pipeline.setup", "--notebook", NOTEBOOK], check=True)

In [ ]:
# Cell 2 of 2: run. Runs the pipeline in the mode set in config/run.yaml, checkpointing to the
# Hub as it goes. A failure is reported to the Hub with its traceback, then raised here.
import os
import subprocess
import sys

os.environ.setdefault("TSFP_WORKDIR", "/content/tsfp_work")
os.chdir("/content/trade-settlement-fail-predictor")
proc = subprocess.Popen(
    [sys.executable, "-u", "-m", "src.pipeline.run_train"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
if proc.wait() != 0:
    subprocess.run([sys.executable, "-m", "src.pipeline.run_report", "--crashed", "--workdir", os.environ["TSFP_WORKDIR"]])
    raise RuntimeError("The run failed. The report is on the Hub: tell Claude Code to check the run.")
print("Finished. Tell Claude Code to check the run.")